# Duka Electronics: Half-Year Sales Exploration (Jan to Jun 2026)

## Executive Summary
- Duka earned **KES 71,792,500** from **1,000 orders** (1 Jan to 29 Jun 2026), **KES 71,792.50** per order.
- **Nairobi** leads with **KES 26,106,100 (36.4%)** from 376 orders, and all five branches rank the same by revenue and by orders because average order values differ by only about 10% (KES 68,353 to 75,591).
- **Laptops** earn **57.9%** of revenue (KES 41,535,000) from 220 orders, and the Premium products (Laptop, Tablet) together earn **84.8%**.
- **Weekends are not a peak:** they hold **27.7%** of orders (277) with a near-identical average order (KES 71,972 vs 71,724), and 5.3 orders a day against 5.6 on weekdays.
- **April** was the best month (KES 13,518,300) and **June** the worst (KES 10,903,400), a **24.0%** gap, and Premium online orders from Nairobi and Mombasa total **83 orders worth KES 10,218,000 (14.2%)**.

## Task 1: Load the data and first-five-minutes inspection
**Approach:** load with `parse_dates`, then check shape, types, missing values, duplicates, date range and categories.

In [1]:
import os, subprocess, sys
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:,.0f}".format); pd.set_option("display.width", 110); pd.set_option("display.max_columns", 20)
if not os.path.exists("duka_sales.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape:", sales.shape, "| missing:", int(sales.isna().sum().sum()),
      "| duplicates:", int(sales.duplicated().sum()), "| order_id unique:", sales["order_id"].is_unique)
print("Types:", sales.dtypes.astype(str).to_dict())
print("Dates:", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
for col in ["branch", "product", "channel"]:
    print(col, sales[col].value_counts().to_dict())

Shape: (1000, 7) | missing: 0 | duplicates: 0 | order_id unique: True
Types: {'order_id': 'int64', 'order_date': 'datetime64[us]', 'branch': 'str', 'product': 'str', 'quantity': 'int64', 'channel': 'str', 'unit_price': 'int64'}
Dates: 2026-01-01 to 2026-06-29
branch {'Nairobi': 376, 'Mombasa': 186, 'Nakuru': 162, 'Kisumu': 159, 'Eldoret': 117}
product {'Laptop': 220, 'Tablet': 215, 'Phone': 194, 'Headphones': 187, 'Charger': 184}
channel {'In-store': 579, 'Online': 320, 'Phone order': 101}


**Inspection notes.** 1,000 rows x 7 columns; `order_date` parsed as datetime; **0 missing values, 0 duplicates**, unique `order_id`; 1 Jan to 29 Jun 2026 (180 days); 5 branches, 5 products, 3 channels. There is no revenue column, so it is derived as quantity x unit price.

## Task 2: Derived columns
**Approach:** `revenue = quantity x unit_price`; `month` and `weekday` from the `dt` accessor; `is_weekend` is Saturday or Sunday; `price_band` is Premium when `unit_price >= 28,000`, otherwise Value.

In [2]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")
months = ["January", "February", "March", "April", "May", "June"]
print("Premium:", sorted(sales.loc[sales["price_band"] == "Premium", "product"].unique()),
      "| total revenue: KES", f"{sales['revenue'].sum():,}")
print(sales[["revenue", "month", "weekday", "is_weekend", "price_band"]].head(2).to_string(index=False))

Premium: ['Laptop', 'Tablet'] | total revenue: KES 71,792,500
 revenue    month   weekday  is_weekend price_band
   45000     June Wednesday       False      Value
  260000 February    Monday       False    Premium


**Answer.** All five columns exist and total revenue is **KES 71,792,500**. Premium is exactly **Laptop and Tablet** (the Tablet's KES 28,000 sits on the boundary, so `>=` matters).

## Task 3: Branch ranking by revenue and by orders
**Approach:** group by branch, rank revenue and order counts, and compare each branch's share and average order value.

In [3]:
branch = (sales.groupby("branch").agg(revenue=("revenue", "sum"), orders=("order_id", "count"),
          avg_order=("revenue", "mean")).sort_values("revenue", ascending=False))
branch["rev_share"] = (branch["revenue"] / branch["revenue"].sum() * 100).map("{:.1f}%".format)
branch["ord_share"] = (branch["orders"] / branch["orders"].sum() * 100).map("{:.1f}%".format)
print("By revenue:", branch.index.tolist())
print("By orders :", branch["orders"].sort_values(ascending=False).index.tolist())
branch.assign(revenue=branch["revenue"].map("{:,}".format))

By revenue: ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret']
By orders : ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret']


            revenue  orders  avg_order rev_share ord_share
branch                                                    
Nairobi  26,106,100     376     69,431     36.4%     37.6%
Mombasa  14,003,700     186     75,289     19.5%     18.6%
Nakuru   11,970,400     162     73,891     16.7%     16.2%
Kisumu   10,868,100     159     68,353     15.1%     15.9%
Eldoret   8,844,200     117     75,591     12.3%     11.7%

**Answer.** **Yes, the rankings are identical.** Revenue is orders x average order, and the average order differs by only about 10% (KES 68,353 in Kisumu to 75,591 in Eldoret) while orders differ more than threefold (376 vs 117), so volume decides the rank. Revenue shares track order shares (Nairobi 36.4% vs 37.6%). Higher average orders only widen some gaps: Nakuru's KES 73,891 average gives it a bigger lead over Kisumu than its 3 extra orders alone would.

## Task 4: Top three products by revenue and by units
**Approach:** group by product, sum revenue and units, take the top three of each and compare with unit price.

In [4]:
product = (sales.groupby("product").agg(unit_price=("unit_price", "first"), units=("quantity", "sum"),
           revenue=("revenue", "sum")))
print("Top 3 by revenue:", product.nlargest(3, "revenue").index.tolist())
print("Top 3 by units  :", product.nlargest(3, "units").index.tolist())
product.sort_values("revenue", ascending=False).assign(revenue=lambda d: d["revenue"].map("{:,}".format))

Top 3 by revenue: ['Laptop', 'Tablet', 'Phone']
Top 3 by units  : ['Tablet', 'Laptop', 'Phone']


            unit_price  units     revenue
product                                  
Laptop           65000    639  41,535,000
Tablet           28000    691  19,348,000
Phone            15000    592   8,880,000
Headphones        2500    555   1,387,500
Charger           1200    535     642,000

**Answer.** By revenue: **Laptop** (KES 41,535,000, 57.9%), **Tablet** (KES 19,348,000) and **Phone** (KES 8,880,000). By units: **Tablet (691)**, Laptop (639) and Phone (592). The same three products appear, but Laptop and Tablet swap places: the Tablet sells 52 more units, yet the Laptop earns 2.1x its revenue because it costs KES 65,000 against KES 28,000. Unit counts are close for all five products (535 to 691), so price, not popularity, drives revenue.

## Task 5: Weekend vs weekday orders
**Approach:** compare orders, total revenue and average order, and also per calendar day, because the period has 128 weekdays but only 52 weekend days.

In [5]:
cal = pd.date_range(sales["order_date"].min(), sales["order_date"].max())
n_we = int((cal.dayofweek >= 5).sum())
wk = (sales.groupby("is_weekend").agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
      avg_order=("revenue", "mean")))
wk.index = ["Weekday", "Weekend"]
wk["days"] = [len(cal) - n_we, n_we]
wk["orders_per_day"] = wk["orders"] / wk["days"]
wk["revenue_per_day"] = wk["revenue"] / wk["days"]
print(f"Weekend share of orders {wk.loc['Weekend', 'orders'] / 1000:.1%} | revenue "
      f"{wk.loc['Weekend', 'revenue'] / wk['revenue'].sum():.1%}")
print(f"Average order {wk.loc['Weekend', 'avg_order'] / wk.loc['Weekday', 'avg_order'] - 1:+.1%} | "
      f"revenue per day {wk.loc['Weekend', 'revenue_per_day'] / wk.loc['Weekday', 'revenue_per_day'] - 1:+.1%}")
wk.assign(revenue=wk["revenue"].map("{:,}".format), orders_per_day=wk["orders_per_day"].map("{:.2f}".format))

Weekend share of orders 27.7% | revenue 27.8%
Average order +0.3% | revenue per day -5.4%


         orders     revenue  avg_order  days orders_per_day  revenue_per_day
Weekday     723  51,856,200     71,724   128           5.65          405,127
Weekend     277  19,936,300     71,972    52           5.33          383,390

**Answer.** Weekday: **723 orders, KES 51,856,200, average KES 71,724**. Weekend: **277 orders, KES 19,936,300, average KES 71,972**. Weekends hold 27.7% of orders and 27.8% of revenue, matching their 28.9% share of days. The average order is virtually the same (+0.3%), and per day weekends are slightly slower (5.3 vs 5.6 orders; KES 383,390 vs 405,127, 5.4% less). There is no weekend peak.

## Task 6: Best and worst month
**Approach:** total revenue per month, then (best - worst) / worst; revenue per calendar day is a check because months differ in length.

In [6]:
monthly = sales.groupby("month").agg(revenue=("revenue", "sum")).reindex(months)
monthly["days"] = pd.Series(cal.month_name()).value_counts().reindex(months).values
monthly["rev_per_day"] = monthly["revenue"] / monthly["days"]
best, worst = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
b, w = monthly.loc[best, "revenue"], monthly.loc[worst, "revenue"]
print(f"Best {best} KES {b:,} | worst {worst} KES {w:,} | best is {(b - w) / w:.1%} higher")
print(f"Per day: best {monthly['rev_per_day'].idxmax()}, lowest {monthly['rev_per_day'].idxmin()}")
print({m: f"{r:,}" for m, r in monthly["revenue"].items()})

Best April KES 13,518,300 | worst June KES 10,903,400 | best is 24.0% higher
Per day: best April, lowest May
{'January': '11,786,600', 'February': '11,772,000', 'March': '12,396,800', 'April': '13,518,300', 'May': '11,415,400', 'June': '10,903,400'}


**Answer.** The best month is **April (KES 13,518,300)** and the worst **June (KES 10,903,400)**: April is **24.0% higher** (June is 19.3% lower). Per calendar day April still leads (KES 450,610) and May is lowest (KES 368,239), so the slowdown after April is real, not an effect of June's 29 days.

## Task 7: Premium online orders from Nairobi and Mombasa
**Approach:** a boolean mask on `price_band`, `channel` and `branch`, exported with `index=False` and read back to confirm every row qualifies.

In [7]:
mask = ((sales["price_band"] == "Premium") & (sales["channel"] == "Online")
        & sales["branch"].isin(["Nairobi", "Mombasa"]))
premium_online = sales[mask]
os.makedirs("outputs", exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
chk = pd.read_csv("outputs/premium_online_coast_capital.csv")
assert (chk["price_band"] == "Premium").all() and (chk["channel"] == "Online").all()
assert set(chk["branch"]) == {"Nairobi", "Mombasa"} and len(chk) == len(premium_online)
print(f"Orders: {len(premium_online)} | revenue: KES {premium_online['revenue'].sum():,} "
      f"({premium_online['revenue'].sum() / sales['revenue'].sum():.1%} of all revenue)")
print(premium_online.groupby("branch")["revenue"].agg(["count", "sum"]).to_dict("index"))

Orders: 83 | revenue: KES 10,218,000 (14.2% of all revenue)
{'Mombasa': {'count': 33, 'sum': 3696000}, 'Nairobi': {'count': 50, 'sum': 6522000}}


**Answer.** **83 orders worth KES 10,218,000 (14.2% of revenue)**: Nairobi 50 orders (KES 6,522,000) and Mombasa 33 (KES 3,696,000), saved to `outputs/premium_online_coast_capital.csv`.

## Task 8: Spreadsheet check (Excel vs pandas)
**Approach:** `excel/duka_branch_summary.xlsx` has a SUMIFS branch ranking, an XLOOKUP region column (my assumed mapping: Nairobi = Capital, Mombasa = Coast, Kisumu = Western, Nakuru and Eldoret = Rift Valley) and a PivotTable of revenue by branch and channel. I compare the saved Excel values with pandas.

In [8]:
xl_path = "excel/duka_branch_summary.xlsx"
xl = pd.read_excel(xl_path, sheet_name="Branch Summary", header=3, nrows=5).set_index("Branch")
piv = pd.read_excel(xl_path, sheet_name="Pivot", header=5, index_col=0, nrows=6)
py = pd.crosstab(sales["branch"], sales["channel"], values=sales["revenue"],
                 aggfunc="sum", margins=True, margins_name="Total Result")
ok_sum = (xl["Total revenue (KES)"] == branch["revenue"].reindex(xl.index)).all()
ok_piv = (piv.to_numpy() == py.to_numpy()).all()
print("SUMIFS matches pandas:", bool(ok_sum), "| PivotTable matches crosstab:", bool(ok_piv))
print(f"Grand total: Excel {piv.iloc[-1, -1]:,} | pandas {sales['revenue'].sum():,}")
assert ok_sum and ok_piv
print("Regions (XLOOKUP):", xl["Region"].to_dict())

SUMIFS matches pandas: True | PivotTable matches crosstab: True
Grand total: Excel 71,792,500 | pandas 71,792,500
Regions (XLOOKUP): {'Nairobi': 'Capital', 'Mombasa': 'Coast', 'Nakuru': 'Rift Valley', 'Kisumu': 'Western', 'Eldoret': 'Rift Valley'}


**Answer.** The Excel totals **match pandas exactly**: SUMIFS revenue for all five branches and every PivotTable cell agree, with a grand total of **KES 71,792,500** in both. In-store is the largest channel in every branch (KES 40,969,200, 57.1%), then Online (KES 22,786,100, 31.7%).

In [9]:
# Final checks: every number quoted in the Executive Summary matches the data
t = sales["revenue"].sum()
assert t == 71_792_500 and len(sales) == 1000
assert branch.index[0] == "Nairobi" and round(branch.loc["Nairobi", "revenue"] / t * 100, 1) == 36.4
assert product.loc["Laptop", "revenue"] == 41_535_000 and round(product.loc["Laptop", "revenue"] / t * 100, 1) == 57.9
assert round(sales.loc[sales["price_band"] == "Premium", "revenue"].sum() / t * 100, 1) == 84.8
assert wk.loc["Weekend", "orders"] == 277 and (best, worst) == ("April", "June")
assert round((b - w) / w * 100, 1) == 24.0 and len(premium_online) == 83
print("All Executive Summary numbers verified.")

All Executive Summary numbers verified.
